In [1]:
import sys 
import os 

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd 
import numpy as np
import unicodedata
import re
from pathlib import Path

from config.data_cleaning_config import (
    DEFAULT_DROP_COLUMNS,
    COLUNAS_TEXTO,
    COLUNAS_BINARIAS,
    COLUNA_DIA_SEMANA,
    COLUNA_MES,
    COLUNA_HORA,
    TESTE
)


In [3]:
data_dir = Path("../data/raw")
arquivos = list(data_dir.glob("*.csv"))

In [4]:
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

## Configuração da base de dados

Ao analiasr manualmente os dados disponíveis em: https://dadosabertos.curitiba.pr.gov.br/conjuntodado/detalhe?chave=b16ead9d-835e-41e8-a4d7-dcc4f2b4b627

observa-se que o banco de dados é cumulativo. Ou seja, o arquivo `2017-02-01_sigesguarda_-_Base_de_Dados` já contém os registros do arquivo `2017-01-01_sigesguarda_-_Base_de_dados`, funcionando como uma mera continuação. Esse padrão se manteve até `2024-02-01_sigesguarda_-_Base_de_Dados.csv`. A partir dessa versão, a coluna `ATENDIMENTO_ANO` foi removida e o esquema do banco de dados foi modificado. 

Diante disso, o objetivo é utilizar o arquivo `2024-02-01_sigesguarda_-_Base_de_Dados.csv` (que abrange dados de 2009 até 12/12/2022) e combiná-lo com a base mais recente disponível: `2026-04-02_sigesguarda_-_Base_de_Dados.csv`. Para isso, é necessário padronizar a base antiga com a base na estrutural atual. As etapas são:

1. Remover a coluna `ATENDIMENTO_ANO` da base antiga;
2. Padronizar a coluna `OCORRENCIA_DATA`. Na base antiga, essa coluna segue o formato `ANO/MÊS/DIA HORA:MINUTO:SEGUNDO`. O mesmo tratamento será aplicado à versão mais recente da base, que utiliza o formato `DIA/MÊS/ANO`.

Após essas padronizações, será feita uma busca por registros duplicados, a fim de identificar o ponto correto de mesclagem entre as duas bases.

## 01. Padronização dos dados de texto

Precisamos de uma função que faça a limpeza dos campos de texto da base de dados com o objetivo de padronizar os valores. Por exemplo, 'Centro Cívico' e 'centro civico' devem ser tratados como iguais. Para isso, iremos remover a acentuação, converter tudo para letras minúsculas e eliminar caracteres especiais, mantendo apenas letras, números e espaços. 

In [5]:
def limpar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    
    valor = unicodedata.normalize("NFKD", valor)
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()
    
    return valor if valor else pd.NA


Essa função irá executar:

1. Tratamento de valores nlos: se o valor for NaN, a função retorna `pd.NA`
2. Conversão para string e limpeza básica: o valor é convertido para string, removeremos espaços extras no início e no fim (`.strip()`) e transformamos tudo em letras minúsculas (.lower())
3. Remoção de acentos: usamos `unicodedata.normalize("NFKD", valor)` para decompor os caracteres acentuados (ex: "ção" vira "c~a~o"), e depois `.encode("ascii", "ignore")` elimina os acentos, mantendo apenas os caracteres ASCII básicos.
4. Filtragem de caracteres: com `re.sub(r"[^a-z0-9\s]", " ", valor)`, substituímos qualquer caractere que não seja letra minúscula, número ou espaço por um espaço em branco. Isso remove pontuação, símbolos e outros caracteres especiais. Esse espaço em branco é normalizado com `re.sub(r"\s+", " ", valor).strip()`, onde é substituida sequência de espaços por um único espaço e `.strip()` remove os espaços desnecessários das bordas. 

Ao fim, se toda a limpeza retornar uma string vazia, retornamos pd.NA. Caso contrário, retornamos o texto limpo.

## 02. Padronização de colunas binárias

Para padronizar colunas binárias, definindo `0` como `False` e `1` como `True`, é necessário primeiro identificar todos os valores distintos presentes na base de dados. Isso garante que o mapeamento seja feito corretamente. Para identificar todos os valores únicos na tupla `COLUNAS_BINARIAS`, disponibilizado em `config/data_cleaning_config.py`, faremos:

In [6]:
valores_unicos = {col: set() for col in COLUNAS_BINARIAS}

In [7]:
for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    
    for col in COLUNAS_BINARIAS:
        if col in df.columns:
            valores = df[col].dropna().unique()
            valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    print(sorted(valores))


FLAG_EQUIPAMENTO_URBANO:
['-----------------------', 'N', 'NÃO', 'SIM', 'Y', 'f', 't']

FLAG_FLAGRANTE:
['--------------', 'NÃO', 'NÃ\x83O', 'SIM']

NATUREZA1_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1', '1.0']

NATUREZA2_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1', '1.0']

NATUREZA3_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1']

NATUREZA4_DEFESA_CIVIL:
['----------------------', '0', '0.0', '1']

NATUREZA5_DEFESA_CIVIL:
['----------------------', '0', '0.0']


In [8]:
def mapear_binario(valor):

    if pd.isna(valor):
        return 0
    
    s = str(valor).strip().lower()

    if s in ('1.0', '0.0'):
        return 1 if s == '1.0' else 0

    valor_limpo = limpar_texto(valor)

    if pd.isna(valor_limpo):
        return 0

    if valor_limpo in ('sim', 'y', 't', '1'):
        return  1
    
    if valor_limpo in ('n', 'nao', 'f', '0', '-----------------------',
                        '--------------', '----------------------',
                        '----------------------', '----------------------', 
                        '----------------------', '----------------------'):
        return 0
    
    return 0

## 03. Mapeamento dos dias da semana

Assim como foi feito para as colunas binárias, iremos encontrar os valores únicos para a coluna `OCORRENCIA_DIA_SEMANA` e fazer o mapeamento para transformar em inteiro.

In [9]:
valores_unicos = {col: set() for col in COLUNA_DIA_SEMANA}

In [10]:
for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    
    for col in COLUNA_DIA_SEMANA:
        if col in df.columns:
            valores = df[col].dropna().unique()
            valores_unicos[col].update(valores)

for col, valores in valores_unicos.items():
    print(f"\n{col}:")
    print(sorted(valores))


OCORRENCIA_DIA_SEMANA:
['---------------------', 'DOMINGO', 'Domingo', 'QUARTA', 'QUINTA', 'Quarta', 'Quinta', 'SEGUNDA', 'SEXTA', 'Segunda', 'Sexta', 'SÁBADO', 'SÃ¡bado', 'Sábado', 'TERÇA', 'TerÃ§a', 'Terça']


In [11]:
def mapear_dia_semana(valor):

    if pd.isna(valor):
        return 0
    
    texto_limpo = limpar_texto(valor)

    if pd.isna(texto_limpo) or texto_limpo == '':
        return 0
    
    mapa_dias = {
        'domingo': 1,
        'segunda': 2,
        'terca': 3,
        'quarta': 4,
        'quinta': 5,
        'sexta': 6,
        'sabado': 7
    }

    return mapa_dias.get(texto_limpo, 0)

## 04. Tratamento da coluna OCORRENCIA_HORA

Iremos padronizar a coluna `OCORRENCIA_HORA` que possui os registros no formato `HH:MM:SS` (hora, minuto e segundo) e, a partir dela, criar as colunas adicionais hora e minuto, descartando os segundos. 

In [12]:
def tratar_coluna_hora(df, coluna):

    if coluna not in df.columns:
        return df 
    
    serie_hora = pd.to_datetime(df[coluna], format="%H:%M:%S", errors="coerce")

    df[coluna] = serie_hora.dt.strftime("%H:%M:%S")

    df[f"{coluna}_HORA"] = serie_hora.dt.hour.astype("Int64")
    df[f"{coluna}_MINUTO"] = serie_hora.dt.minute.astype("Int64")

    df.drop(columns=[coluna], inplace=True)
    
    return df 

## 05. Flags: MANHA, TARDE, NOITE, MADRUGADA

Vamos criar quatro colunas binária, indicando em qual período do dia a hora de uma ocorrência se enquadrada: madrugada, manhã, tarde ou noite. Cada coluna recebe o valor `1` (verdadeiro) ou `0` (falso). Para isso, utilizaremos os intervalos:

- Madrugada: 00:00 - 05:59
- Manhã: 06:00 - 11:59
- Tarde: 12:00 - 17:59
- Noite: 18:00 - 23:59

In [13]:
def adicionar_periodo_dia(df, coluna_hora):

    if coluna_hora not in df.columns:
        return df 
    
    horas = pd.to_numeric(df[coluna_hora], errors="coerce")

    df["MADRUGADA"] = ((horas >= 0) & (horas <= 5)).astype("Int64")
    df["MANHA"] = ((horas >= 6) & (horas <= 11)).astype("Int64")
    df["TARDE"] = ((horas >= 12) & (horas <= 17)).astype("Int64")
    df["NOITE"] = ((horas >= 18) & (horas <= 23)).astype("Int64")

    return df

## 06. Analisar colunas faltantes

Vamos verificar se todas as fontes de dados possuem as mesmas colunas. Para isso, utilizaremos o arquivo `2017-01-01_sigesguarda_-_Base_de_Dados.csv` como referência e extrairemos os cabeçalhos dos demais arquivos CSV, a fim de identificar possíveis colunas faltantes.

In [14]:
def ler_cabecalho(caminho):
    df = pd.read_csv(caminho, nrows=0, encoding="utf-8", sep=";")
    return list(df.columns)

In [15]:
cabecalhos = {arq: ler_cabecalho(arq) for arq in arquivos}

In [16]:
nome_referencia = "2017-01-01_sigesguarda_-_Base_de_Dados.csv"
caminho_referencia = data_dir / nome_referencia

In [17]:
ref_colunas = cabecalhos[caminho_referencia]

In [18]:
for arquivo, colunas in cabecalhos.items():
    if arquivo == caminho_referencia:
        continue 
    if colunas != ref_colunas:
        print(f"Diferença encontrada em: {arquivo.name}")
        print("Colunas faltando:", set(ref_colunas) - set(colunas))
        print("Colunas extras:", set(colunas) - set(ref_colunas))
        print("-" * 50)

Diferença encontrada em: 2024-04-24_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: set()
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-07-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-12-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-07-09_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------
Diferença encontrada em: 2024-09-01_sigesguarda_-_Base_de_Dados.csv
Colunas faltando: {'ATENDIMENTO_ANO'}
Colunas extras: set()
--------------------------------------------------


Os arquivos a seguir não possuem a coluna 'ATENDIMENTO_ANO':

- `2024-04-24_sigesguarda_-_Base_de_Dados`; 
- `2024-07-01_sigesguarda_-_Base_de_Dados`;
- `2024-12-01_sigesguarda_-_Base_de_Dados`; 
- `2024-07-09_sigesguarda_-_Base_de_Dados`; 
- `2024-09-01_sigesguarda_-_Base_de_Dados`;

Para correção, considerando que o ano de atendimento corresponde ao ano de disponibilização da base, propõe-se criar a coluna 'ATENDIMENTO_ANO' e preenchê-la com o valor 2024 para todos os registros desses arquivos.

## 07. Pipeline de limpeza de dados

In [ ]:
def executar_pipeline(arquivo):
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)

    for col in COLUNAS_TEXTO:
        if col in df.columns:
            df[col] = df[col].apply(limpar_texto)
        
    for col in COLUNAS_BINARIAS:
        if col in df.columns:
            df[col] = df[col].apply(mapear_binario).astype("Int64")

    for col in COLUNA_DIA_SEMANA:
        if col in df.columns:
            df[col] = df[col].apply(mapear_dia_semana).astype("Int64")

    df = tratar_coluna_hora(df, COLUNA_HORA)
    df = adicionar_periodo_dia(df, f"{COLUNA_HORA}_HORA")

    colunas_para_remover = [col for col in DEFAULT_DROP_COLUMNS if col in df.columns]
    if colunas_para_remover:
        df = df.drop(columns=colunas_para_remover)
    
    caminho_saida = output_dir / arquivo.name 
    df.to_csv(caminho_saida, sep=";", index=False, encoding="utf-8")

In [20]:
for arquivo in sorted(arquivos):
    executar_pipeline(arquivo)